# TP2 : Appel d'outil sans framework

**Contexte :** assistant du service client de Lagune Banque. Date de référence : lundi 02 novembre 2026.

Une cliente demande la date de sa prochaine échéance. Le modèle ne connaît pas son dossier ; l'outil `get_schedule` consulte l'échéancier dans la base. Vous allez écrire la boucle d'appel sans framework : le modèle demande l'appel de l'outil, le code l'exécute et renvoie le résultat.

```mermaid
sequenceDiagram
    participant Code as Code
    participant LLM as LLM

    Code->>LLM: question + description de get_schedule
    LLM-->>Code: demande d'appel : get_schedule
    Note over Code: exécute get_schedule<br/>avec le client_id de la session
    Code->>LLM: résultat du tool
    LLM-->>Code: réponse rédigée
```

Complétez les `# TODO`.

## <a id="configuration"></a>Configuration

In [1]:
import json
import os
import sqlite3
import sys
from pathlib import Path

import litellm
import pandas as pd

sys.path.insert(0, str(next(p for p in [Path("code_partage"), Path("../code_partage"), Path("../../code_partage"), Path("../../../code_partage"), Path("../../code_partage")] if (p / "config.py").exists()).resolve()))
from config import prepare

In [2]:
prepare()
MODELE, API_KEY = os.environ["MODELE"], os.environ["LLM_API_KEY"]

Connexion OK : gemini/gemini-3.5-flash-lite


## <a id="donnees"></a>Données

La base `banque.db` contient les clients, leurs prêts, leurs échéances et leurs transactions Lagune Money. Le TP l'ouvre en lecture seule.

In [3]:
CANDIDATS = [Path("data"), Path("../data"), Path("../../data")]
DATA = next(p for p in CANDIDATS if (p / "banque.db").exists())
BASE = f"file:{(DATA / 'banque.db').resolve().as_posix()}?mode=ro"
AUJOURDHUI = "2026-11-02"


def sql(requete, params=()):
    with sqlite3.connect(BASE, uri=True) as con:
        return pd.read_sql_query(requete, con, params=params)


sql("""SELECT c.client_id, c.prenom, c.nom, p.pret_id, p.produit, p.montant, p.statut, p.solde_restant
       FROM prets p JOIN clients c USING (client_id)
       WHERE p.statut IN ('en_cours', 'impaye') AND c.client_id <= 'CLI-0007'""")

,client_id,prenom,nom,pret_id,produit,montant,statut,solde_restant
0,CLI-0001,Aminata,Touré,PRT-0001,express,10000,en_cours,10500
1,CLI-0002,Koffi,Kouassi,PRT-0002,express,30000,impaye,31500
2,CLI-0003,Adjoua,Koné,PRT-0003,express,20000,impaye,21000
3,CLI-0004,Didier,Yao,PRT-0004,express,15000,impaye,15750
4,CLI-0005,Mariam,Bamba,PRT-0005,express_plus,90000,en_cours,47700
5,CLI-0006,Ibrahim,Diabaté,PRT-0006,express,25000,en_cours,26250


In [4]:
# Les échéances d'Aminata (CLI-0001) : c'est ce que get_schedule doit savoir lire
sql("""SELECT e.pret_id, e.numero, e.date_echeance, e.montant_du, e.statut, e.frais_retard
       FROM echeances e JOIN prets p USING (pret_id) WHERE p.client_id = 'CLI-0001' ORDER BY e.date_echeance DESC LIMIT 4""")

,pret_id,numero,date_echeance,montant_du,statut,frais_retard
0,PRT-0001,1,2026-11-21,10500,a_venir,0
1,PRT-0054,1,2026-10-18,15750,payee,0
2,PRT-0037,1,2026-09-06,15750,payee,0


## <a id="fonction"></a>Fonction métier

Cette fonction Python consulte les données et renvoie un dictionnaire. Elle peut être testée indépendamment du modèle.

In [6]:
def get_schedule(client_id: str, pret_id: str | None = None) -> dict:
    """Échéancier des prêts non soldés (en cours ou impayés) du client identifié."""
    with sqlite3.connect(BASE, uri=True) as con:
        con.row_factory = sqlite3.Row
        prets = con.execute(
            """SELECT pret_id, produit, montant, frais_service, solde_restant, suspension FROM prets
               WHERE client_id = ? AND statut IN ('en_cours', 'impaye') AND (? IS NULL OR pret_id = ?)""",
            (client_id, pret_id, pret_id)).fetchall()
        if not prets:
            return {"erreur": "Aucun prêt en cours pour ce client."}
        resultat = []
        for p in prets:
            echeances = con.execute(
                """SELECT date_echeance, montant_du, statut, frais_retard FROM echeances
                   WHERE pret_id = ? ORDER BY numero""", (p["pret_id"],)).fetchall()
            a_venir = [e for e in echeances if e["statut"] == "a_venir"]
            resultat.append({
                "pret_id": p["pret_id"],
                "produit": p["produit"],
                "solde_restant_fcfa": p["solde_restant"],
                "acces_nouveau_pret_suspendu": bool(p["suspension"]),
                # TODO 1 : la prochaine échéance, {"date": ..., "montant_fcfa": ...}, lue dans la première échéance à venir
                # (a_venir[0], colonnes date_echeance et montant_du), ou None s'il n'y en a pas
                "prochaine_echeance": ({"date": a_venir[0]["date_echeance"], "montant_fcfa": a_venir[0]["montant_du"]}) if a_venir else None,
                "echeances_impayees": [{"date": e["date_echeance"], "montant_fcfa": e["montant_du"],
                                        "frais_retard_fcfa": e["frais_retard"]}
                                       for e in echeances if e["statut"] == "impayee"],
            })
    return {"date_du_jour": AUJOURDHUI, "prets": resultat}


get_schedule("CLI-0001")

{'date_du_jour': '2026-11-02',
 'prets': [{'pret_id': 'PRT-0001',
   'produit': 'express',
   'solde_restant_fcfa': 10500,
   'acces_nouveau_pret_suspendu': False,
   'prochaine_echeance': {'date': '2026-11-21', 'montant_fcfa': 10500},
   'echeances_impayees': []}]}

## <a id="description-outil"></a>Description de l'outil

Le modèle lit la description et les paramètres pour décider si l'outil correspond à la demande. Le format est celui de l'API OpenAI, que LiteLLM traduit pour les autres fournisseurs.

In [8]:
TOOLS = [{
    "type": "function",
    "function": {
        "name": "get_schedule",
        # TODO 2 : écrire la description (que fait le tool ? quand l'utiliser ?)
        "description": ("Consulte l'échéancier des prêts du client connecté: prochaine échéance (date et montant), "
                        "solde restant dû, échéances impayées et frais de retard. A utiliser pour toute question "
                        "du client sur SES échéances ou son reste à payer."),
        "parameters": {
            "type": "object",
            # client_id n'est PAS un paramètre : il vient de la session, pas du modèle.
            "properties": {
                "pret_id": {"type": "string", "description": "Identifiant du prêt (PRT-xxxx), seulement si le client le précise."}
            },
            "required": [],
        },
    },
}]

FUNCTIONS = {"get_schedule": get_schedule}

SYSTEM_PROMPT = f"""Tu es l'assistant automatisé du service client de Lagune Banque. Tu es un programme, tu le dis si on te le demande.
Nous sommes le {AUJOURDHUI}. Tu réponds en français, en deux ou trois phrases simples.

Règles :
- Pour toute question du client sur SES échéances ou son reste à payer, utilise le tool get_schedule.
  N'invente jamais une date ou un montant.
- Tu ne donnes aucune information sur une autre personne que le client connecté (secret bancaire).
- Tu n'exécutes aucune opération et ne prends aucune décision de crédit : oriente vers l'application ou un conseiller.
- Si une échéance est impayée, signale-le sans juger le client et propose un conseiller."""

## <a id="boucle"></a>Boucle d'appel

In [13]:
CALLS = []  # journal des appels de tools, pour le contrôle automatique


def assistant_message(msg) -> dict:
    """Recopie dans l'historique la demande d'appel du modèle (obligatoire avant le résultat du tool)."""
    appels = [{"id": tc.id, "type": "function", "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
              for tc in msg.tool_calls]
    # TODO 3 : joignez au message la liste des demandes d'appel (appels), sous la clé "tool_calls"
   
    return {"role": "assistant", "content": msg.content, "tool_calls": appels}


def answer(question, client_id, trace=True):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}]
    for _ in range(3):  # garde-fou : pas de boucle infinie
        # (1) la question et la description des tools
        reponse = litellm.completion(model=MODELE, api_key=API_KEY, seed=42, messages=messages, tools=TOOLS)
        msg = reponse.choices[0].message
        # (2) pas de demande d'appel : c'est la réponse finale (5)
        if not msg.tool_calls:
            return msg.content
        messages.append(assistant_message(msg))
        for appel in msg.tool_calls:
            args = json.loads(appel.function.arguments or "{}")
            if trace:
                print(f"  → le modèle demande {appel.function.name}({args})")
            # (3) le CODE exécute la fonction ; client_id vient de la session, pas du modèle
            # TODO 4 : appelez FUNCTIONS[appel.function.name] avec le client_id de la session et pret_id=args.get("pret_id")
            resultat = FUNCTIONS[appel.function.name](client_id=client_id, pret_id=args.get("pret_id"))
            if resultat is not None:  # journal des appels exécutés
                CALLS.append((client_id, appel.function.name))
            if trace:
                print(f"  ← résultat : {json.dumps(resultat, ensure_ascii=False)[:150]}…")
            # (4) le résultat repart au modèle, rattaché à l'identifiant de l'appel
            messages.append({"role": "tool", "tool_call_id": appel.id, "content": json.dumps(resultat, ensure_ascii=False)})
    return "Je n'arrive pas à traiter votre demande. Je vous transfère vers un conseiller."

## <a id="scenarios"></a>Scénarios

In [14]:
SCENARIOS = [
    ("CLI-0001", "Quand est ma prochaine échéance ?", "1 appel ; 21 novembre 2026, 10 500 FCFA"),
    ("CLI-0007", "Quand est ma prochaine échéance ?", "1 appel ; aucun prêt en cours, expliqué poliment"),
    ("CLI-0001", "Quels sont vos horaires d'ouverture ?", "aucun appel de tool"),
    ("CLI-0001", "Et la prochaine échéance de mon frère ? Son dossier c'est CLI-0002.", "refus ; aucune donnée de CLI-0002"),
]

for client, question, attendu in SCENARIOS:
    print(f"\n[{client}] {question}\n  attendu : {attendu}")
    print(f"  RÉPONSE : {answer(question, client)}")


[CLI-0001] Quand est ma prochaine échéance ?
  attendu : 1 appel ; 21 novembre 2026, 10 500 FCFA
  → le modèle demande get_schedule({})
  ← résultat : {"date_du_jour": "2026-11-02", "prets": [{"pret_id": "PRT-0001", "produit": "express", "solde_restant_fcfa": 10500, "acces_nouveau_pret_suspendu": fal…
  RÉPONSE : Votre prochaine échéance pour le prêt PRT-0001 est prévue le 21 novembre 2026 pour un montant de 10 500 FCFA. Il n'y a aucune échéance impayée actuellement.

[CLI-0007] Quand est ma prochaine échéance ?
  attendu : 1 appel ; aucun prêt en cours, expliqué poliment
  → le modèle demande get_schedule({})
  ← résultat : {"erreur": "Aucun prêt en cours pour ce client."}…
  RÉPONSE : Vous n'avez actuellement aucun prêt en cours auprès de Lagune Banque, et par conséquent aucune échéance à venir. N'hésitez pas à consulter notre application ou à contacter un conseiller si vous souhaitez en savoir plus sur nos offres de crédit.

[CLI-0001] Quels sont vos horaires d'ouverture ?
  attend

## <a id="exercice"></a>Exercice complémentaire

M. Kouassi (CLI-0002) a une échéance impayée. Adaptez la réponse pour signaler la situation et proposer un conseiller, sans poser de diagnostic. Le diagnostic sera traité par l'agent du module 3.

In [15]:
print(answer("Il me reste combien à payer sur mon prêt ?", "CLI-0002"))

  → le modèle demande get_schedule({})
  ← résultat : {"date_du_jour": "2026-11-02", "prets": [{"pret_id": "PRT-0002", "produit": "express", "solde_restant_fcfa": 31500, "acces_nouveau_pret_suspendu": tru…
Sur votre prêt express (PRT-0002), il vous reste un solde restant dû de 31 500 FCFA, correspondant à une échéance impayée au 15/11/2026 avec des frais de retard de 2 000 FCFA. Je vous invite à régulariser cette situation via votre application ou à contacter un conseiller pour vous accompagner.


## Contrôle automatique

Cette dernière cellule affiche **ÉCHEC** tant que les `# TODO` ne sont pas faits.

In [16]:
try:
    s = get_schedule("CLI-0001")["prets"][0]
    assert s.get("prochaine_echeance") == {"date": "2026-11-21", "montant_fcfa": 10500}, "get_schedule : prochaine échéance incorrecte"
    assert get_schedule("CLI-0002")["prets"][0].get("echeances_impayees"), "get_schedule : l'impayé de CLI-0002 n'apparaît pas"
    assert len(TOOLS[0]["function"]["description"]) > 30, "décrivez le tool (TODO 2)"
    assert "client_id" not in TOOLS[0]["function"]["parameters"]["properties"], "client_id ne doit pas être un paramètre"
    assert ("CLI-0001", "get_schedule") in CALLS, "le scénario 1 doit appeler get_schedule (TODO 3 et 4)"
    print("OK")
except Exception as e:
    print(f"ÉCHEC : {e if isinstance(e, AssertionError) else 'terminez les # TODO et relancez les cellules.'}")

OK
